In [ ]:
# %load ../../notebooks/init.ipy
%reload_ext autoreload
%autoreload 2

# Builtin packages
from datetime import datetime
from importlib import reload
import logging
import os
from pathlib import Path
import sys
import warnings

# standard secondary packages
import astropy as ap
import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import scipy.stats
import tqdm.notebook as tqdm

# development packages
import kalepy as kale
import kalepy.utils
import kalepy.plot

# --- Holodeck ----
import holodeck as holo
from holodeck import cosmo, utils, plot
from holodeck.constants import MSOL, PC, YR, MPC, GYR, SPLC, NWTG
import holodeck.sams
import holodeck.gravwaves
#import holodeck.evolution
#import holodeck.population

# Silence annoying numpy errors
np.seterr(divide='ignore', invalid='ignore', over='ignore')
warnings.filterwarnings("ignore", category=UserWarning)

# Plotting settings
mpl.rc('font', **{'family': 'serif', 'sans-serif': ['Times'], 'size': 15})
mpl.rc('lines', solid_capstyle='round')
mpl.rc('mathtext', fontset='cm')
mpl.style.use('default')   # avoid dark backgrounds from dark theme vscode
plt.rcParams.update({'grid.alpha': 0.5})

# Load log and set logging level
log = holo.log
# log.setLevel(logging.INFO)
#log.setLevel(logging.DEBUG)
#log.setLevel(log.DEBUG)
#log.level, log.DEBUG, log.INFO

In [ ]:
def get_sam_dadt(nrads=100, shape=None, gsmf_flag=2, gpf_flag=0, tau=1.0, 
                       ainit=1.0e4, rc=100.0, nu_in=-1.0, nu_out=+2.5):
    if gpf_flag: 
        _gpf = sams.GPF_Power_Law()
    else:
        _gpf = None
    
    if gsmf_flag == 1:
        _gsmf = holo.sams.GSMF_Schechter()
    elif gsmf_flag == 2:
        _gsmf = holo.sams.GSMF_Double_Schechter()
    else: 
        raise ValueError('keyword `gsmf_flag` must be 1 for single Schecter or 2 for double Schechter')
        
        
    sam = holo.sams.Semi_Analytic_Model(
        #mtot = (1.0e4*MSOL, 1.0e12*MSOL, 91),
        mtot = (1.0e5*MSOL, 1.0e11*MSOL, 91),
        #mtot = (1.0e9*MSOL, 1.0e10*MSOL, 91),
        shape = shape,
        gsmf = _gsmf,
        gpf = _gpf
    )
    hard = holo.hardening.Fixed_Time_2PL_SAM(
        sam, tau * GYR,
        sepa_init = ainit * PC,
        rchar = rc * PC,
        gamma_inner = nu_in,
        gamma_outer = nu_out
    )

    
    # () start from the hardening model's initial separation
    rmax = hard._sepa_init
    # (M,) end at the ISCO
    rmin = utils.rad_isco(sam.mtot)
    # rmin = hard._TIME_TOTAL_RMIN * np.ones_like(sam.mtot)
    # Choose steps for each binary, log-spaced between rmin and rmax
    extr = np.log10([rmax * np.ones_like(rmin), rmin])
    radii = np.linspace(0.0, 1.0, nrads)[np.newaxis, :]
    # (M, X)
    radii = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * radii
    radii = 10.0 ** radii
    # (M, Q, Z, X)
    mt, mr, rz, rads = np.broadcast_arrays(
        sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
        sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
        sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
        radii[:, np.newaxis, np.newaxis, :]
    )
    # (X, M*Q*Z)
    #mt, mr, rz, rads = [mm.reshape(-1, STEPS).T for mm in [mt, mr, rz, rads]]
    #print(f'{sam.mtot.shape=}, {sam.mrat.shape=}, {sam.redz.shape=}, {radii.shape=}')
    #print(f'{mt.shape=}, {mr.shape=}, {rz.shape=}, {rads.shape=}')
    #print('Mtot=', sam.mtot/MSOL)
    #print('q=', sam.mrat)
    #print('redz=', sam.redz)

    # old: (X, M*Q*Z) --- `Fixed_Time.dadt` will only accept this shape
    # new: (M, Q, Z, X) is shape of input and output arrays
    dadt = hard.dadt(mt, mr, rads)
    print(f"{mt.shape=}, {mr.shape=}, {dadt.shape=}, {rads.shape=}")

    return sam, hard, rads, dadt
    
def sepa_emit(mtot, fgw):
    """
    separation of an equal-mass circular binary with total mass mtot emitting GWs at frequency fgw
    
    assumes mtot in cgs and fgw in Hz, returns separation in cm
    """
    #print(f'{MSOL=}, {mtot=}, {fgw=}, {NWTG=}')
    return ( NWTG * mtot / (fgw * np.pi) **2 )**(1.0/3)

def calc_and_plot_dadt(sam_data, distance_units='pc'):
    
    #if gpf_flag: 
    #    _gpf = sams.GPF_Power_Law()
    #else:
    #    _gpf = None
    
    #if gsmf_flag == 1:
    #    _gsmf = holo.sams.GSMF_Schechter()
    #elif gsmf_flag == 2:
    #    _gsmf = holo.sams.GSMF_Double_Schechter()
    #else: 
    #    raise ValueError('keyword `gsmf_flag` must be 1 for single Schecter or 2 for double Schechter')
           
    #sam = holo.sams.Semi_Analytic_Model(
    #    shape=shape,
    #    gsmf = _gsmf,
    #    gpf = _gpf
    #)
    #hard = holo.hardening.Fixed_Time_2PL_SAM(
    #    sam, tau * GYR,
    #    sepa_init = ainit * PC,
    #    rchar = rc * PC,
    #    gamma_inner = nu_in,
    #    gamma_outer = nu_out
    #)
    
    ## () start from the hardening model's initial separation
    #rmax = hard._sepa_init
    ## (M,) end at the ISCO
    #rmin = utils.rad_isco(sam.mtot)
    ## rmin = hard._TIME_TOTAL_RMIN * np.ones_like(sam.mtot)
    ## Choose steps for each binary, log-spaced between rmin and rmax
    #extr = np.log10([rmax * np.ones_like(rmin), rmin])
    #radii = np.linspace(0.0, 1.0, nrads)[np.newaxis, :]
    ## (M, X)
    #radii = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * radii
    #radii = 10.0 ** radii
    ## (M, Q, Z, X)
    #mt, mr, rz, rads = np.broadcast_arrays(
    #    sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
    #    sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
    #    sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
    #    radii[:, np.newaxis, np.newaxis, :]
    #)
    # (X, M*Q*Z)
    ##mt, mr, rz, rads = [mm.reshape(-1, STEPS).T for mm in [mt, mr, rz, rads]]
    ##print(f'{sam.mtot.shape=}, {sam.mrat.shape=}, {sam.redz.shape=}, {radii.shape=}')
    ##print(f'{mt.shape=}, {mr.shape=}, {rz.shape=}, {rads.shape=}')

    ## (X, M*Q*Z) --- `Fixed_Time.dadt` will only accept this shape
    #dadt = hard.dadt(mt, mr, rads)
    #print(f"{dadt.shape=}")

    
    nskip = 1
    if distance_units == 'pc':
        xlim=[1e-8,1e5]
    elif distance_units == 'rg':
        xlim=[1,1e13]
    else:
        raise ValueError(f"invalid keyword {distance_units=}. must be 'pc' or 'rg'.")
    
    # Make the plot
    fig = plt.figure(figsize=(12,9))
    
    ax1 = fig.add_subplot(231)
    plt.xscale('log')
    plt.xlim(xlim[0],xlim[1])
    plt.yscale('log')
    ax1.xaxis.set_inverted(True)
    plt.xlabel(f'binary separation [{distance_units}]')
    plt.ylabel('hardening tscale [yr]')
    #plt.ylim(1e4,1e10)

    ax2 = fig.add_subplot(232)
    plt.xscale('log')
    plt.xlim(xlim[0],xlim[1])
    plt.yscale('log')
    ax2.xaxis.set_inverted(True)
    plt.xlabel(f'binary separation [{distance_units}]')
    plt.ylabel('hardening rate [cm/s]')
    #plt.ylim(1e4,1e10)

    ax3 = fig.add_subplot(233)
    plt.xscale('log')
    plt.xlim(xlim[0],xlim[1])
    plt.yscale('log')
    ax3.xaxis.set_inverted(True)
    plt.xlabel(f'binary separation [{distance_units}]')
    plt.ylabel('cumulative time [yr]')

    ax4 = fig.add_subplot(234)
    plt.xscale('log')
    plt.xlim(xlim[0],xlim[1])
    plt.yscale('log')
    ax4.xaxis.set_inverted(True)
    plt.xlabel(f'a_GW,crit [{distance_units}]')
    plt.ylabel('time from fobs=1/30yr to ISCO [yr]')

    ax5 = fig.add_subplot(235)
    plt.xscale('log')
    plt.xlim(xlim[0],xlim[1])
    plt.yscale('log')
    ax5.xaxis.set_inverted(True)
    plt.xlabel(f'a_GW,crit [{distance_units}]')
    plt.ylabel('a(fobs=1/30yr) / a_GW,crit')

    
    cmap_arr = ['Blues', 'Oranges',  'Greens', 'Reds', 'Purples',
                'Greys', 'YlOrBr', 'YlOrRd', 'OrRd', 'PuRd', 'RdPu', 'BuPu',
                'GnBu', 'PuBu', 'YlGnBu', 'PuBuGn', 'BuGn', 'YlGn']*2
    fgw_ls = ['-','-.',':']
    lhandles = []
    flhandles = []
    
    nu_mrk = ['s','^','o','*']
    
    for n,sd in enumerate(sam_data):
        sam, hard, rads, dadt = sd

        agw = calc_aGW_for_Fixed_Time_2PL(hard, sam)
        print(f"{hard._norm.shape=} {agw.shape=}")
        
        print(f"{rads[0,0,0,0]=}, {rads[0,0,0,-1]=}")
        times_evo = calc_cumulative_thard(sd, rads[0,0,0,0], rads[0,0,0,-1])
        #times_evo = -utils.trapz_loglog(-1.0 / dadt[:,:,0,:], rads[:,:,0,:], axis=2, cumsum=True)
         
        if n==0:
            print('Mtot=', sam.mtot/MSOL)
            print('q=', sam.mrat)
            print('redz=', sam.redz)

        cmap = plot._get_cmap(cmap_arr[n])
        colors = cmap(np.linspace(0.3, 1, sam.mtot.size))
        lw = np.arange(0.5,sam.mrat.size, 0.5)
    
        for i in np.arange(0,sam.mtot.size,nskip):
            
            if distance_units == 'pc':
                dunits = PC
            elif distance_units == 'rg':
                dunits = NWTG * sam.mtot[i] / SPLC**2
            
            #print(f'Mtot = {sam.mtot[i]/MSOL:.2g}')
            for fi,fem in enumerate([0.1,0.3,1.0]):
                sepa_em = sepa_emit(sam.mtot[i], fem/YR) / dunits
                fl,= ax1.plot([sepa_em,sepa_em],[1e-4,1e10],ls=fgw_ls[fi],
                              alpha=0.5,color=colors[i],label=f'fgw={fem}/yr')
                #ax2.plot([sepa_em,sepa_em],[1e-3,1e11],ls=fgw_ls[fi],color=colors[i])
                #print(f'{fi=},{fgw_ls[fi]}, {fem}')
                if i==sam.mtot.size-1 and n==len(sam_data)-1:
                    flhandles += [fl]
                    
            a_late = sepa_emit(sam.mtot[i], 1.0/(30*YR)) 
            #print(f"mtot={sam.mtot[i]/MSOL:.3g}: in {distance_units}: {a_late/dunits=:.3g} "
            #      f"{agw[i,0]/dunits=:.3g} {agw[i,-1]/dunits=:.3g} {a_late/agw[i,0]=:.3g} {a_late/agw[i,-1]=:.3g}")
            #print(f"{sam.mtot[i]=}: {a_late/dunits=} [{distance_units}]")
            times_early = calc_cumulative_thard(sd, rads[0,0,0,0], a_late)
            times_late = calc_cumulative_thard(sd, a_late, rads[0,0,0,-1])
            #print(f"{times_evo.shape=},{times_early.shape=},{times_late.shape=}")
 
            for j in np.arange(0,sam.mrat.size,nskip):
                l,= ax1.plot(rads[i,j,0,:]/dunits, -rads[i,j,0,:]/dadt[i,j,0,:]/YR, 
                             alpha=0.5, color=colors[i], lw=lw[j], 
                             label=f'tau={hard._target_time/GYR}')
                if i==sam.mtot.size-1 and j==sam.mrat.size-1:
                    lhandles += [l]
                if j==sam.mrat.size-1 and n==len(sam_data)-1:
                    ax2.plot(rads[i,j,0,:]/dunits, -dadt[i,j,0,:], 
                             alpha=0.5, color=colors[i], lw=lw[j], 
                             label=f'mtot={sam.mtot[i]/MSOL:.2g}')
                else:
                    ax2.plot(rads[i,j,0,:]/dunits, -dadt[i,j,0,:], 
                             alpha=0.5, color=colors[i], lw=lw[j],label=None)
                #print(f"plotting {agw[i,j]=} for mtot={sam.mtot[i]/MSOL}, mrat={sam.mrat[j]}")
                #ax2.plot([agw[i,j]/dunits, agw[i,j]/dunits], [1e-3,1e11], color=colors[i], alpha=0.1)
                
                
                ax3.plot(rads[i,j,0,:-1]/dunits, times_evo[i,j,:]/YR, 
                         alpha=0.5, color=colors[i], lw=lw[j])
                #print(f"{times_early[i,j,:].size=}, {times_late[i,j,:].size=}")
                #nt_early = times_early[i,j,:].size
                #nt_late = times_late[i,j,:].size
                #ax3.plot(rads[i,j,0,:nt_early]/dunits, times_early[i,j,:]/YR, 
                #         alpha=0.5, color='r', lw=lw[j])
                #ax3.plot(rads[i,j,0,nt_early+1:-1]/dunits, (times_early[i,j,-1]+times_late[i,j,:])/YR, 
                #         alpha=0.5, color='b', lw=lw[j])
                
            ax2.plot([a_late/dunits, a_late/dunits], [1e-3,1e11], color=colors[i], alpha=0.1)
            #print(f"{times_early[:,:,-1].flatten().shape}")
            #print(f"{times_late[:,:,-1].flatten().shape}")
            ##ax4.scatter(agw.flatten()/dunits, times_early[:,:,-1].flatten()/YR,color=colors[i],alpha=0.5)
            ax4.scatter(agw[i,:]/dunits, times_late[i,:,-1]/YR, color=colors[i],
                        alpha=0.5,marker=nu_mrk[int(2*hard._gamma_inner+3)])
            #print(f"{a_late/agw[i,:]=}")
            ax5.scatter(agw[i,:]/dunits, a_late/agw[i,:], color=colors[i],
                        alpha=0.5,marker=nu_mrk[int(2*hard._gamma_inner+3)])
            ##ax4.scatter(np.tile(a_late/dunits,times_late[i,:,-1].shape[0]), times_late[i,:,-1]/YR,color=colors[i],
            ##            alpha=0.5,marker='.')
            #ax4.plot([a_late/dunits, a_late/dunits], [1e4,1e10],alpha=0.3,color=colors[i])
        ax1.plot(xlim, [hard._target_time/YR, hard._target_time/YR], '--', color='darkgray')
        if distance_units=='pc': ax1.plot([hard._rchar/dunits, hard._rchar/dunits], [1e-2,1e10],'k--')
        if distance_units=='pc': ax2.plot([hard._rchar/dunits, hard._rchar/dunits], [10,1e11],'k--')
    ax2.plot(xlim, [3.0e10,3.0e10], color='magenta')
    leg2 = ax1.legend(handles=flhandles, loc='lower right')
    #ax1.legend(handles=lhandles, loc='lower left')
    ax1.add_artist(leg2)
    ax2.legend(loc='lower left')
        
    plt.suptitle(f'ai={hard._sepa_init/PC:.2g}pc, '
                 f'rc={hard._rchar/PC:.2g}pc,'
                 f' nu_in={hard._gamma_inner}, nu_out={hard._gamma_outer}\n'
                 f'Mtot=({sam.mtot.min()/MSOL:.2g},{sam.mtot.max()/MSOL:.2g})Msun, '
                 f'q=({sam.mrat.min():.2g},{sam.mrat.max():.2g})')
    fig.subplots_adjust(wspace=0.3,top=0.85, right=0.95)
    

In [ ]:
def calc_cumulative_thard(_sam_data, astart, astop):

    _sam, _hard, _rads, _dadt = _sam_data

    idx_avals = np.where((_rads[0,0,0,:]<=astart)&(_rads[0,0,0,:]>=astop))[0]
    
    _tevo = -utils.trapz_loglog(-1.0 / _dadt[:,:,0,idx_avals], _rads[:,:,0,idx_avals], 
                                axis=2, cumsum=True)
    return _tevo


def calc_aGW_for_Fixed_Time_2PL(_hard, _sam):
    """Calculate (circular) binary separation of transition from 'inner' power-law hardening to GW regime"""

    
    _mt, _mr = np.broadcast_arrays(
        _sam.mtot[:, np.newaxis],
        _sam.mrat[np.newaxis, :]
    )
    
    _m1, _m2 = utils.m1m2_from_mtmr(_mt, _mr)
    #print(f"{mt/MSOL=},{mr=}")
    #print(f"{m1/MSOL=},{m2/MSOL=}")
    dadt_gw_const = - (64/5.0) * NWTG**3 / SPLC**5 * _m1 * _m2 * _mt
    #print(f"{dadt_gw_const.shape=} {dadt_gw_const=}")
    dadt_innerPL_const = - _hard._norm * _hard._rchar**(_hard._gamma_inner-1)
    #print(f"{dadt_innerPL_const.shape=} {dadt_innerPL_const=}")
    
    aGW = ( dadt_gw_const / dadt_innerPL_const ) ** (1.0/(4.0-_hard._gamma_inner))
    #print(f"{aGW=}")
    
    return aGW

#def calc_aGW_for_Fixed_Time_2PL(norm, nu_inner, rchar, mtot, mrat):
#    """Calculate (circular) binary separation of transition from 'inner' power-law hardening to GW regime"""
#
#    
#    mt, mr = np.broadcast_arrays(
#        mtot[:, np.newaxis],
#        mrat[np.newaxis, :]
#    )
#    
#    m1, m2 = utils.m1m2_from_mtmr(mt, mr)
#    #print(f"{mt/MSOL=},{mr=}")
#    #print(f"{m1/MSOL=},{m2/MSOL=}")
#    dadt_gw_const = - (64/5.0) * NWTG**3 / SPLC**5 * m1 * m2 * mt
#    #print(f"{dadt_gw_const.shape=} {dadt_gw_const=}")
#    dadt_innerPL_const = - norm * rchar**(nu_inner-1)
#    #print(f"{dadt_innerPL_const.shape=} {dadt_innerPL_const=}")
#    
#    aGW = ( dadt_gw_const / dadt_innerPL_const ) ** (1.0/(4.0-nu_inner))
#    #print(f"{aGW=}")
#    
#    return aGW

In [ ]:
nr = 100
sh = 2
gsmff = 2
gpff = 0
ai = 1.0e4
rch=10.0
t=1.0
nin=-1.0
nout=0.0
sam, hard, rads, dadt = get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, gpf_flag=gpff, 
                                     ainit=ai, tau=t, rc=rch, nu_in=nin, nu_out=nout)

print(hard._norm.shape, sam.mtot.shape, sam.mrat.shape)
print(hard._norm, sam.mtot/MSOL, sam.mrat)
#agw_test = calc_aGW_for_Fixed_Time_2PL(hard._norm, hard._gamma_inner, hard._rchar, sam.mtot, sam.mrat)
agw_test = calc_aGW_for_Fixed_Time_2PL(hard, sam)
print(agw_test.shape)
print(agw_test/PC)

times_evo = -utils.trapz_loglog(-1.0 / dadt[:,:,0,:], rads[:,:,0,:], axis=2, cumsum=True)
print(f"{dadt.shape=}, {rads.shape=}, {times_evo.shape=}")
tt = times_evo/GYR
#tt = times_evo[-1, :]/GYR
fig, ax = plot.figax(scale='log')
ax.xaxis.set_inverted(True)
print(utils.stats(tt))
#kale.dist1d(tt, density=True)
for m in range(rads.shape[0]):
    for q in range(rads.shape[1]):
        plt.plot(rads[m,q,0,:-1]/PC,tt[m,q,:])
plt.show()

In [ ]:
#calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=1.0, 
#                   ainit=1.0e4, rc=100.0, nu_in=-1.0, nu_out=+2.5)

nr = 100
sh = 2
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
#for t in [0.1,1.0,10.0]:
for t in [0.1,1.0]:
    for rch in [10,100]:
        for nin in [-1.0,0.0]:
            for nout in [2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
#calc_and_plot_dadt(sam_data)
calc_and_plot_dadt(sam_data, distance_units='rg')
calc_and_plot_dadt(sam_data, distance_units='pc')


In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [0.1,10]:
    for rch in [10,100]:
        for nin in [-1.5,0.0]:
            for nout in [0.0,+2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data)

In [ ]:
1/(30*YR)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [1]:
    for rch in [10,100.0]:
        for nin in [-1.0]:
            for nout in [0.0,+2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [0.1,1.0,10.0]:
    for rch in [10]:
        for nin in [-1.0]:
            for nout in [2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [0.1,1.0,10.0]:
    for rch in [100]:
        for nin in [-1.0]:
            for nout in [2.5]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [0.1,1.0,10.0]:
    for rch in [10]:
        for nin in [-1.0]:
            for nout in [1.0]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data)

In [ ]:
nr = 100
sh = 4
gsmff = 2
gpff = 0
ai = 1.0e4
sam_data = []
for t in [0.1,1.0,10.0]:
    for rch in [100]:
        for nin in [-0.5]:
            for nout in [2.0]:
                sam_data = sam_data + [(get_sam_dadt(nrads=nr, shape=sh, gsmf_flag=gsmff, 
                                                     gpf_flag=gpff, ainit=ai, 
                                                     tau=t, rc=rch, nu_in=nin, nu_out=nout))]

print(len(sam_data))
print(len(sam_data[0]))
calc_and_plot_dadt(sam_data)

In [ ]:
calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=3.0, 
                   ainit=1.0e4, rc=100.0, nu_in=-1.0, nu_out=+1.5)

In [ ]:
calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=1.0, 
                   ainit=1.0e4, rc=0.0, nu_in=-1.0, nu_out=+2.5)

In [ ]:
calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=1.0, 
                   ainit=1.0e4, rc=10.0, nu_in=-1.0, nu_out=+2.5)

calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=3.0, 
                   ainit=1.0e3, rc=10.0, nu_in=-1.0, nu_out=+2.5)

In [ ]:
calc_and_plot_dadt(nrads=100, shape=10, gsmf_flag=2, gpf_flag=0, tau=1.0, 
                   ainit=1.0e3, rc=10.0, nu_in=-1.0, nu_out=+2.5)

In [ ]:
STEPS = 100

# () start from the hardening model's initial separation
rmax = hard._sepa_init
# (M,) end at the ISCO
rmin = utils.rad_isco(sam.mtot)
# rmin = hard._TIME_TOTAL_RMIN * np.ones_like(sam.mtot)
# Choose steps for each binary, log-spaced between rmin and rmax
extr = np.log10([rmax * np.ones_like(rmin), rmin])
radii = np.linspace(0.0, 1.0, STEPS)[np.newaxis, :]
# (M, X)
radii = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * radii
radii = 10.0 ** radii
# (M, Q, Z, X)
mt, mr, rz, rads = np.broadcast_arrays(
    sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
    sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
    sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
    radii[:, np.newaxis, np.newaxis, :]
)
# (X, M*Q*Z)
#mt, mr, rz, rads = [mm.reshape(-1, STEPS).T for mm in [mt, mr, rz, rads]]
print(f'{sam.mtot.shape=}, {sam.mrat.shape=}, {sam.redz.shape=}, {radii.shape=}')
print(f'{mt.shape=}, {mr.shape=}, {rz.shape=}, {rads.shape=}')

# (X, M*Q*Z) --- `Fixed_Time.dadt` will only accept this shape
dadt = hard.dadt(mt, mr, rads)
print(f"{dadt.shape=}")


In [ ]:
##pta_dur = 16.03 * YR
#nfreqs = 40
##hifr = nfreqs/pta_dur
##pta_cad = 1.0 / (2 * hifr)
#fobs_cents = holo.utils.nyquist_freqs(pta_dur, pta_cad)
#fobs_edges = holo.utils.nyquist_freqs_edges(pta_dur, pta_cad)

nreals = 10
nloud = 1

freqs, freqs_edges = utils.pta_freqs()
gwb = sam.gwb(freqs, hard, realize=nreals, loudest=nloud, params=True)

In [ ]:
print(utils.stats(rads[-1,-1,-1,:]/dadt[-1,-1,-1,:]/GYR))
plt.xscale('log')
plt.yscale('log')
nskip = 1
cmap = plot._get_cmap('viridis')
colors = cmap(np.linspace(0, 1, int(sam.mtot.size/nskip)+1))
lw = np.arange(0,int(sam.mrat.size/nskip)+1, 0.1)

for i in np.arange(0,sam.mtot.size,nskip):
    for j in np.arange(0,sam.mrat.size,nskip):
        plt.plot(rads[i,j,0,:]/PC, -rads[i,j,0,:]/dadt[i,j,0,:]/GYR, 
                 alpha=0.5, color=colors[i], lw=lw[j])

In [ ]:
tt = times_evo[-1,-1,-1, :]/GYR
fig, ax = plot.figax(scale='lin')
print(utils.stats(tt))
kale.dist1d(tt, density=True)
plt.show()

In [ ]:
nbins = [5, 10, 123, 0]
_, fit_lamp, fit_plaw, fit_med_lamp, fit_med_plaw = holo.librarian.fit_spectra(fobs_cents, gwb, nbins=nbins)

In [ ]:
num_snaps = len(nbins)
fig, axes = plt.subplots(figsize=[10, 5], ncols=2)
for med, fits, ax in zip([fit_med_lamp, fit_med_plaw], [fit_lamp, fit_plaw], axes):
    for ii, nn in enumerate(nbins):
        if np.all(fits[:, ii] == 0.0):
            continue
        color = ax._get_lines.get_next_color()
        kale.dist1d(fits[:, ii], ax=ax, label=str(nn), color=color)
        ax.axvline(med[ii], ls='--', color=color)
    
    ax.legend()
    
plt.show()


In [ ]:
fig = plot.plot_gwb(fobs_cents, gwb)
ax = fig.axes[0]

xx = fobs_cents * YR
yy = 1e-15 * np.power(xx, -2.0/3.0)
ax.plot(xx, yy, 'r-', alpha=0.5, lw=1.0, label="$10^{-15} \cdot f_\\mathrm{yr}^{-2/3}$")

fits = holo.librarian.get_gwb_fits_data(fobs_cents, gwb)

for ls, idx in zip([":", "--"], [1, -1]):
    med_lamp = fits['fit_med_lamp'][idx]
    med_plaw = fits['fit_med_plaw'][idx]
    yy = (10.0 ** med_lamp) * (xx ** med_plaw)
    label = fits['fit_nbins'][idx]
    label = 'all' if label in [0, None] else label
    ax.plot(xx, yy, color='k', ls=ls, alpha=0.5, lw=2.0, label=str(label) + " bins")

label = fits['fit_label'].replace(" | ", "\n")
fig.text(0.99, 0.99, label, fontsize=6, ha='right', va='top')

ax.legend()
plt.show()


In [ ]:
fig = plot.plot_gwb(fobs_cents, gwb)
ax = fig.axes[0]

xx = fobs_cents * YR
yy = np.median(gwb, axis=-1)
ax.plot(xx, yy, 'k:')

for nn in [5, 10, None]:
    xx, amp, gamma = holo.librarian.fit_powerlaw(fobs_cents, np.median(gwb, axis=-1), nn)
    ax.plot(xx, amp * (xx ** gamma), ls='--')

plt.show()


In [ ]:
fig = plot.plot_gwb(fobs_cents, gwb, nsamp=None)
ax = fig.axes[0]

xx = fobs_cents * YR
yy = np.median(gwb, axis=-1)
ax.plot(xx, yy, 'k-')

nreals = gwb.shape[1]

fits = np.zeros((nreals, 2))
for nn in range(nreals):
    yy = gwb[:, nn]
    xx, *fits[nn, :] = holo.librarian.fit_powerlaw(fobs_cents, yy, 5)
    cc, = ax.plot(xx, fits[nn, 0] * (xx ** fits[nn, 1]), ls='--', alpha=0.5)
    cc = cc.get_color()
    ax.plot(fobs_cents*YR, yy, color=cc, alpha=0.5)

plt.show()

draw_fits = fits.copy()
draw_fits[:, 0] = np.log10(draw_fits[:, 0])

kale.corner(draw_fits.T)
plt.show()


In [ ]:
hard_time=-2.2957907176750907
hard_gamma_inner=-1.3335554512862717
gsmf_phi0=-2.802178096487384
gsmf_mchar0=11.704311872442908
gsmf_alpha0=-1.7179504809027346
gpf_zbeta=2.397456708546681
gpf_qgamma=0.4609649227136603
gmt_norm=0.5765308121579338
gmt_zbeta=-0.26777937808636665
mmb_amp=8.301258575486393
mmb_plaw=0.4785954601355894
mmb_scatter=0.12386778329303819

hard_time = (10.0 ** hard_time) * GYR
gmt_norm = gmt_norm * GYR
mmb_amp = (10.0 ** mmb_amp) * MSOL

gsmf = holo.sam.GSMF_Schechter(phi0=gsmf_phi0, mchar0_log10=gsmf_mchar0, alpha0=gsmf_alpha0)
gpf = holo.sam.GPF_Power_Law(qgamma=gpf_qgamma, zbeta=gpf_zbeta)
gmt = holo.sam.GMT_Power_Law(time_norm=gmt_norm, zbeta=gmt_zbeta)
mmbulge = holo.host_relations.MMBulge_KH2013(mamp=mmb_amp, mplaw=mmb_plaw, scatter_dex=mmb_scatter)

sam = holo.sam.Semi_Analytic_Model(
    gsmf=gsmf, gpf=gpf, gmt=gmt, mmbulge=mmbulge,
    shape=20
)
hard = holo.hardening.Fixed_Time.from_sam(
    sam, hard_time, gamma_sc=hard_gamma_inner,
    progress=False
)
pta_dur = 16.03 * YR
nfreqs = 40
hifr = nfreqs/pta_dur
pta_cad = 1.0 / (2 * hifr)
fobs_cents = holo.utils.nyquist_freqs(pta_dur, pta_cad)
fobs_edges = holo.utils.nyquist_freqs_edges(pta_dur, pta_cad)
gwb = sam.gwb(fobs_edges, realize=10, hard=hard)

plot.plot_gwb(fobs_cents, gwb)
plt.show()


In [ ]:
#SHAPE = None
SHAPE = 30
TIME = 1.0 * GYR

sam = holo.sam.Semi_Analytic_Model(shape=SHAPE)
hard = holo.hardening.Fixed_Time.from_sam(sam, TIME, interpolate_norm=False)

In [ ]:
STEPS = 100

# () start from the hardening model's initial separation
rmax = hard._sepa_init
# (M,) end at the ISCO
rmin = utils.rad_isco(sam.mtot)
# rmin = hard._TIME_TOTAL_RMIN * np.ones_like(sam.mtot)
# Choose steps for each binary, log-spaced between rmin and rmax
extr = np.log10([rmax * np.ones_like(rmin), rmin])
rads = np.linspace(0.0, 1.0, STEPS)[np.newaxis, :]
# (M, X)
rads = extr[0][:, np.newaxis] + (extr[1] - extr[0])[:, np.newaxis] * rads
rads = 10.0 ** rads
# (M, Q, Z, X)
mt, mr, rz, rads = np.broadcast_arrays(
    sam.mtot[:, np.newaxis, np.newaxis, np.newaxis],
    sam.mrat[np.newaxis, :, np.newaxis, np.newaxis],
    sam.redz[np.newaxis, np.newaxis, :, np.newaxis],
    rads[:, np.newaxis, np.newaxis, :]
)
# (X, M*Q*Z)
mt, mr, rz, rads = [mm.reshape(-1, STEPS).T for mm in [mt, mr, rz, rads]]
# (X, M*Q*Z) --- `Fixed_Time.dadt` will only accept this shape
dadt = hard.dadt(mt, mr, rads)
# Integrate (inverse) hardening rates to calculate total lifetime to each separation
times_evo = -utils.trapz_loglog(-1.0 / dadt, rads, axis=0, cumsum=True)


In [ ]:
tt = times_evo[-1, :]/GYR
fig, ax = plot.figax(scale='lin')
print(utils.stats(tt))
kale.dist1d(tt, density=True)
plt.show()